In [ ]:
import os
import csv
import ast
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib as mpl
import sys
sys.path.insert(0, str(Path("..").resolve()))
from utils import savefig

plt.rcParams['font.size'] = 14

mpl.rcParams['svg.fonttype'] = 'none'

save_fig = False

In [ ]:
gamma_names = ["0", "02", "04", "06", "08", "10"]
temporal_discount_factors = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
noise_levels = [0, 0.2, 0.4, 0.6, 0.8, 1]
noise_names = ["0", "02", "04", "06", "08", "1"]

In [ ]:
df = pd.read_pickle("../data/df_models.pkl")
df_reservoir = pd.read_pickle("../data/df_reservoir.pkl")
df_tcm = pd.read_pickle("../data/df_tcm.pkl")
df_perf = pd.read_pickle("../data/df_perf.pkl")

### clustering

In [ ]:
df_for_clustering = df[(df["accuracy"] >= 0.6) & (df["temporal_factor"] >= 0.5)].copy()
print(len(df_for_clustering))

In [ ]:
""" test best number of clusters """

from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer, PowerTransformer, normalize
from sklearn.cluster import KMeans  
from sklearn.metrics import silhouette_score

features = ["forward_asymmetry", "temporal_factor",
            "index_decoding_accuracy", "item_decoding_accuracy", "last_item_decoding_accuracy",
            "explained_variance_index", "explained_variance_identity", 
            "cross_decoding_accuracy_index", "cross_decoding_accuracy_identity",
            ]

X = df_for_clustering[features]

# scaler = StandardScaler()
X_normalized = X

# Determine the optimal number of clusters with elbow method
inertia = []
silhoutte_score = []
for n_clusters in range(1, 11):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(X_normalized)
    labels = kmeans.labels_
    inertia.append(kmeans.inertia_)
    if n_clusters > 1:
        silhoutte_score.append(silhouette_score(X_normalized, labels))

plt.figure(figsize=(4, 3.3), dpi=180)
plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
if save_fig:
    savefig("./figures/supp3", "suppfig3A", format="svg", close=False)
plt.show()

plt.figure(figsize=(4, 3.3), dpi=180)
plt.plot(range(2, 11), silhoutte_score, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette score')
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
if save_fig:
    savefig("./figures/supp3", "suppfig3B", format="svg", close=False)
plt.show()

In [ ]:
""" cluster based on the chosen number of clusters """
# make the optimal number of clusters

# Fit the KMeans model with the optimal number of clusters
optimal_clusters = 3
kmeans = KMeans(n_clusters=optimal_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(X_normalized)
df_for_clustering["cluster"] = cluster_labels

# Add silhouette score to evaluate clustering quality
silhouette_avg = silhouette_score(X_normalized, cluster_labels)
print(f'Silhouette Score for {optimal_clusters} clusters: {silhouette_avg}')



In [ ]:
# print number of items for each cluster
for i in range(optimal_clusters):
    print(f'Cluster {i+1}: {len(df_for_clustering[df_for_clustering["cluster"] == i])} items')
# print 5 info for each cluster
for i in range(optimal_clusters):
    print(f'Cluster {i+1}:')
    print(df_for_clustering[df_for_clustering['cluster'] == i][["temporal_discount_factor", "noise_level", "model_num"]].tail())

In [ ]:
df_filtered = df_for_clustering

In [ ]:
# change number of clusters
df_filtered["cluster"] = df_filtered["cluster"].replace({0: 2, 2: 0})

In [ ]:
""" t-SNE visualization of clusters """

from sklearn.manifold import TSNE

colormap = np.array(["#2A9D8F", "#E9C46A", "#E76F51", "#F4A261", "#264653"]) 

X_tsne = df_filtered[features]

X_tsne_normalized = X_tsne
cluster_labels = df_filtered["cluster"]

# Perform t-SNE on the normalized data
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_tsne_normalized)

# Plot the t-SNE results with cluster labels
plt.figure(figsize=(3, 3.3), dpi=200)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=colormap[cluster_labels], alpha=0.5)
plt.xlabel('t-SNE component 1')
plt.ylabel('t-SNE component 2')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
if save_fig:
    savefig("./figures/fig2", "fig2A", format="svg", close=False)
plt.show()

In [ ]:
# save data with cluster labels
# df_filtered.to_pickle("../data/df_filtered_cluster.pkl")

### plot relationship of metrics with clusters

In [ ]:
# plot scatter with cluster labels

def plot_scatter_cluster(df, x, y, c, x_label, y_label, fit=True, plot_reservoir=True, save_folder=None, save_name=None):
    plt.figure(figsize=(3.4, 3.3), dpi=200)

    plt.scatter(df[x], df[y], c=colormap[df[c].iloc[:]], alpha=0.5)

    if fit:
        if plot_reservoir:
            # plot for reservoir and tcm
            plt.scatter(df_reservoir[x], df_reservoir[y], c=colormap[-1], alpha=0.5)
    
        # Reshape x and y for sklearn
        x_reshaped = np.array(df[x]).reshape(-1, 1)
        y_reshaped = np.array(df[y]).reshape(-1, 1)

        # Fit linear regression model
        model = LinearRegression()
        model.fit(x_reshaped, y_reshaped)

        # Predict y values
        sorted_x_idx = np.argsort(x_reshaped.reshape(-1))
        x1, x2 = x_reshaped[sorted_x_idx[0]], x_reshaped[sorted_x_idx[-1]]
        y_pred = model.predict(x_reshaped)
        y1, y2 = y_pred[sorted_x_idx[0]], y_pred[sorted_x_idx[-1]]

        # Calculate R-squared
        r2 = r2_score(y_reshaped, y_pred)
        # Calculate the p-value for the regression
        _, p_value = stats.pearsonr(x_reshaped.flatten(), y_reshaped.flatten())
        
        # Plot the regression line
        plt.plot([x1, x2], [y1, y2], color="grey")

        x_label = x_label # +"\n(R² = {:.2f}, p = {:.3f})".format(r2, p_value)
        print("freedom:", x_reshaped.shape[0]-2, "r2: ", r2, "p_value: ", p_value)

        ax = plt.gca()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.xlabel(x_label)
    plt.ylabel(y_label)

    plt.tight_layout()

    if save_fig and save_folder and save_name:
        savefig(save_folder, save_name, format="svg", close=False)

    # plt.tight_layout()
    plt.show()


In [ ]:
plot_scatter_cluster(df_filtered, 'forward_asymmetry', 'temporal_factor', 'cluster', 'Forward asymmetry', 'Temporal organization score')
plot_scatter_cluster(df_filtered, 'explained_variance_index', 'explained_variance_identity', 'cluster', 'Variance explained\n(index)', 'Variance explained\n(identity)')
plot_scatter_cluster(df_filtered, 'cross_decoding_accuracy_index', 'cross_decoding_accuracy_identity', 'cluster', 'Cross decoding accuracy\n(index)', 'Cross decoding accuracy\n(identity)')

plot_scatter_cluster(df_filtered, 'forward_asymmetry', 'explained_variance_index', 'cluster', 'Forward asymmetry', 'Variance explained (index)')
plot_scatter_cluster(df_filtered, 'forward_asymmetry', 'cross_decoding_accuracy_index', 'cluster', 'Forward asymmetry', 'Cross decoding accuracy\n(index)')

plot_scatter_cluster(df_filtered, 'temporal_factor', 'explained_variance_index', 'cluster', 'Temporal organization score', 'Variance explained (index)')
plot_scatter_cluster(df_filtered, 'temporal_factor', 'cross_decoding_accuracy_index', 'cluster', 'Temporal organization score', 'Cross decoding accuracy\n(index)')


### cluster the performance dataset

In [ ]:
from sklearn.cluster import KMeans

features = ["forward_asymmetry", "temporal_factor",
            "index_decoding_accuracy", "item_decoding_accuracy", "last_item_decoding_accuracy",
            "explained_variance_index", "explained_variance_identity", 
            "cross_decoding_accuracy_index", "cross_decoding_accuracy_identity",
            ]

df_perf_filtered = df_perf.copy()

X_perf = df_perf_filtered[features]

X_perf_normalized = X_perf

cluster_labels_perf = kmeans.predict(X_perf_normalized)

df_perf_filtered["cluster"] = cluster_labels_perf
df_perf_filtered["cluster"] = df_perf_filtered["cluster"].replace({0: 2, 2: 0})

print(cluster_labels_perf)

print(df_perf_filtered["cluster"].value_counts())






In [ ]:
# df_perf_filtered.to_pickle("../data/df_perf_cluster.pkl")

In [ ]:
plot_scatter_cluster(df_perf_filtered, 'forward_asymmetry', 'accuracy', 'cluster', 'Forward asymmetry', 'Task accuracy', fit=False)

### cluster for all data

In [ ]:
X_all = df[features]
X_all_normalized = X_all

cluster_labels = kmeans.predict(X_all_normalized)

df["cluster"] = cluster_labels

print(df["cluster"].value_counts())



In [ ]:
# distribution of strategies in each factor

def plot_bar_by_strategy(df, x, x_label, save_folder=None, save_name=None):
    plt.figure(figsize=(3.2, 3.3), dpi=180)
    
    cluster_distribution = df.groupby(["cluster", x]).size().unstack(fill_value=0)
    sum_distribution = cluster_distribution.sum(axis=0)
    cluster_distribution = cluster_distribution.div(sum_distribution, axis=1)
    cumsum_distribution = cluster_distribution.cumsum(axis=0)

    for i, row in cumsum_distribution.iterrows():
        plt.bar(row.index, row.values, color=colormap[i], label=f"Cluster {i}", width=0.12, zorder=-i)
    plt.xlabel(x_label)
    plt.ylabel("Distribution of strategies")

    unique_x = df[x].unique()
    plt.xticks(unique_x, unique_x)
    
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()

    if save_fig and save_folder and save_name:
        savefig(save_folder, save_name, format="svg", close=False)

    plt.show()


In [ ]:
df_all_tdf = df_filtered[(df_filtered["noise_level"] == 1.0)]
plot_bar_by_strategy(df_all_tdf, "temporal_discount_factor", "Discount factor", save_folder="./figures/fig3", save_name="fig3F")

df_all_noise = df_filtered[df_filtered["temporal_discount_factor"] == 1.0]
plot_bar_by_strategy(df_all_noise, "noise_level", "% WM flushed", save_folder="./figures/fig3", save_name="fig3G")

